In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import pandas as pd

# --- 1. Настройка SparkSession ---
spark = SparkSession.builder \
    .appName("DB_Connection_Check") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.2.19") \
    .getOrCreate()

print("✅ Spark Session успешно создана.")

# --- 2. Параметры подключения ---
JDBC_URL = "jdbc:postgresql://host.docker.internal:5432/postgres"
USER = "postgres"
PASSWORD = "masterkey" 
DRIVER = "org.postgresql.Driver"

jdbc_properties = {
    "user": USER,
    "password": PASSWORD,
    "driver": DRIVER
}

# ----------------------------------------------------
# 2. ЗАГРУЗКА ТРЕБУЕМЫХ ТАБЛИЦ
# ----------------------------------------------------
try:
    # Загружаем таблицу с авторами 
    df_actor = spark.read.jdbc(url=JDBC_URL, table="actor", properties=jdbc_properties)
    print("Загружена таблица 'actor'.")

    # Загружаем таблицу адресами
    df_address = spark.read.jdbc(url=JDBC_URL, table="address", properties=jdbc_properties)
    print("Загружена таблица 'address'.")

     # Загружаем таблицу с названиями категорий
    df_category = spark.read.jdbc(url=JDBC_URL, table="category", properties=jdbc_properties)
    print("Загружена таблица 'category'.")

    # Загружаем таблиц с гордами 
    df_city = spark.read.jdbc(url=JDBC_URL, table="city", properties=jdbc_properties)
    print("Загружена таблица 'city'.")

     # Загружаем таблицу с странами
    df_country = spark.read.jdbc(url=JDBC_URL, table="country", properties=jdbc_properties)
    print("Загружена таблица 'country'.")

    # Загружаем таблицу кастомерами 
    df_customer = spark.read.jdbc(url=JDBC_URL, table="customer", properties=jdbc_properties)
    print("Загружена таблица 'customer'.")

     # Загружаем таблицу с названиями фильмов
    df_film = spark.read.jdbc(url=JDBC_URL, table="film", properties=jdbc_properties)
    print("Загружена таблица 'film'.")

    # Загружаем таблицу-связку фильм-актер
    df_film_actor = spark.read.jdbc(url=JDBC_URL, table="film_actor", properties=jdbc_properties)
    print("Загружена таблица 'film_actor'.")

    # Загружаем таблицу-связку фильм-категория
    df_film_category = spark.read.jdbc(url=JDBC_URL, table="film_category", properties=jdbc_properties)
    print("Загружена таблица 'film_category'.")

     # Загружаем таблицу с inventory
    df_inventory = spark.read.jdbc(url=JDBC_URL, table="inventory", properties=jdbc_properties)
    print("Загружена таблица 'inventory'.")

    # Загружаем таблицу с языками
    df_language = spark.read.jdbc(url=JDBC_URL, table="language", properties=jdbc_properties)
    print("Загружена таблица 'language'.")

     # Загружаем таблицу с названиями категорий
    df_payment = spark.read.jdbc(url=JDBC_URL, table="payment", properties=jdbc_properties)
    print("Загружена таблица 'payment'.")

    # Загружаем таблицу арендой 
    df_rental = spark.read.jdbc(url=JDBC_URL, table="rental", properties=jdbc_properties)
    print("Загружена таблица 'rantal'.")

     # Загружаем таблицу с персоналом 
    df_staff = spark.read.jdbc(url=JDBC_URL, table="staff", properties=jdbc_properties)
    print("Загружена таблица 'staff'.")

    # Загружаем таблицу магазинов 
    df_store = spark.read.jdbc(url=JDBC_URL, table="store", properties=jdbc_properties)
    print("Загружена таблица 'store'.")

except Exception as e:
    print(f"ОШИБКА: Не удалось загрузить таблицы из БД. Подробности: {e}")
    spark.stop()
    raise e



✅ Spark Session успешно создана.
Загружена таблица 'actor'.
Загружена таблица 'address'.
Загружена таблица 'category'.
Загружена таблица 'city'.
Загружена таблица 'country'.
Загружена таблица 'customer'.
Загружена таблица 'film'.
Загружена таблица 'film_actor'.
Загружена таблица 'film_category'.
Загружена таблица 'inventory'.
Загружена таблица 'language'.
Загружена таблица 'payment'.
Загружена таблица 'rantal'.
Загружена таблица 'staff'.
Загружена таблица 'store'.


In [5]:
# Выведите количество фильмов в каждой категории, отсортированное по убыванию. 
df_joined = df_film_category.join(  
    df_category,
    df_film_category["category_id"] == df_category["category_id"], 
    "inner"
)
print("Объединение (JOIN) таблиц выполнено.")

df_result = df_joined.groupBy(col("name").alias("category_name")) \
                     .agg(count(col("film_id")).alias("film_count"))

print("Группировка по категории и подсчет фильмов завершен.")

df_final = df_result.orderBy(col("film_count").desc())

print("\nРЕЗУЛЬТАТ: Количество фильмов в каждой категории (по убыванию):\n")
df_final.show(truncate=False)

Объединение (JOIN) таблиц выполнено.
Группировка по категории и подсчет фильмов завершен.

РЕЗУЛЬТАТ: Количество фильмов в каждой категории (по убыванию):

+-------------+----------+
|category_name|film_count|
+-------------+----------+
|Sports       |74        |
|Foreign      |73        |
|Family       |69        |
|Documentary  |68        |
|Animation    |66        |
|Action       |64        |
|New          |63        |
|Drama        |62        |
|Games        |61        |
|Sci-Fi       |61        |
|Children     |60        |
|Comedy       |58        |
|Travel       |57        |
|Classics     |57        |
|Horror       |56        |
|Music        |51        |
+-------------+----------+



In [6]:
# Выведите 10 актеров, фильмы которых были в прокате чаще всего, отсортированных по убыванию. 
df_joined_1 = df_rental.join(
    df_inventory,
    on="inventory_id", 
    how="inner"
)

df_joined_2 = df_joined_1.join(
    df_film_actor,
    on="film_id",
    how="inner"
)

df_final_join = df_joined_2.join(
    df_actor,
    on="actor_id",
    how="inner"
)
print("Объединение четырех таблиц завершено.")

df_grouped = df_final_join.groupBy("actor_id", "first_name", "last_name") \
                          .agg(count(col("rental_id")).alias("rental_count"))

df_sorted = df_grouped.orderBy(col("rental_count").desc())

df_top_10 = df_sorted.limit(10)

df_result = df_top_10.withColumn(
    "actor_name",
    concat_ws(" ", col("first_name"), col("last_name"))
).select("actor_name", "rental_count")

print("Агрегация, сортировка и ограничение Топ-10 завершены.")

print("РЕЗУЛЬТАТ: Топ-10 актеров, фильмы которых арендовали чаще всего:\n")
df_result.show(truncate=False)

Объединение четырех таблиц завершено.
Агрегация, сортировка и ограничение Топ-10 завершены.
РЕЗУЛЬТАТ: Топ-10 актеров, фильмы которых арендовали чаще всего:

+------------------+------------+
|actor_name        |rental_count|
+------------------+------------+
|GINA DEGENERES    |753         |
|MATTHEW CARREY    |678         |
|MARY KEITEL       |674         |
|ANGELA WITHERSPOON|654         |
|WALTER TORN       |640         |
|HENRY BERRY       |612         |
|JAYNE NOLTE       |611         |
|VAL BOLGER        |605         |
|SANDRA KILMER     |604         |
|SEAN GUINESS      |599         |
+------------------+------------+



In [7]:
# Выведите категорию фильмов, на которые было потрачено больше всего денег. 

df_joined_1 = df_film.join(
    df_film_category,
    on="film_id", 
    how="inner"
)

df_final_join = df_joined_1.join(
    df_category,
    on="category_id",
    how="inner"
)
print("Объединение трех таблиц завершено.")

df_grouped = df_final_join.groupBy("name") \
                          .agg(sum(col("replacement_cost")).alias("total_replacement_cost"))

df_sorted = df_grouped.orderBy(col("total_replacement_cost").desc())

df_top_1 = df_sorted.limit(1)

print("Группировка, суммирование и поиск TOP-1 завершены.")

print("\nРЕЗУЛЬТАТ: Категория фильмов, на которую было потрачено больше всего денег:")

df_top_1.show(truncate=False)

Объединение трех таблиц завершено.
Группировка, суммирование и поиск TOP-1 завершены.

РЕЗУЛЬТАТ: Категория фильмов, на которую было потрачено больше всего денег:
+------+----------------------+
|name  |total_replacement_cost|
+------+----------------------+
|Sports|1509.26               |
+------+----------------------+



In [10]:
# Вывести названия фильмов, которых нет в инвентаре. 

df_not_in_inventory = df_film.join(
    df_inventory,
    on="film_id",
    how="left_anti"
).select("title") 

print("Объединение LEFT ANTI JOIN завершено.")

print("\nРЕЗУЛЬТАТ: Названия фильмов, которых нет в инвентаре:")

df_not_in_inventory.show(df_not_in_inventory.count(), truncate=False)

Объединение LEFT ANTI JOIN завершено.

РЕЗУЛЬТАТ: Названия фильмов, которых нет в инвентаре:
+----------------------+
|title                 |
+----------------------+
|CHOCOLATE DUCK        |
|BUTCH PANTHER         |
|VOLUME HOUSE          |
|ORDER BETRAYED        |
|TADPOLE PARK          |
|KILL BROTHERHOOD      |
|FRANKENSTEIN STRANGER |
|CROSSING DIVORCE      |
|SUICIDES SILENCE      |
|CATCH AMISTAD         |
|PERDITION FARGO       |
|FLOATS GARDEN         |
|GUMP DATE             |
|WALLS ARTIST          |
|GLADIATOR WESTWARD    |
|HOCUS FRIDA           |
|ARSENIC INDEPENDENCE  |
|MUPPET MILE           |
|FIREHOUSE VIETNAM     |
|ROOF CHAMPION         |
|DAZED PUNK            |
|PEARL DESTINY         |
|RAINBOW SHOCK         |
|KENTUCKIAN GIANT      |
|BOONDOCK BALLROOM     |
|COMMANDMENTS EXPRESS  |
|HATE HANDICAP         |
|ARK RIDGEMONT         |
|CROWDS TELEMARK       |
|DELIVERANCE MULHOLLAND|
|RAIDERS ANTITRUST     |
|SISTER FREDDY         |
|VILLAIN DESPERATE     |
|APOLLO

In [11]:
# Выведите тройку актёров, снявшихся в фильмах категории «Дети» чаще всего. 
# Если у нескольких актёров одинаковое количество фильмов, выведите их всех. 

df_joined_1 = df_film_actor.join(
    df_film_category,
    on="film_id", 
    how="inner"
)

df_joined_2 = df_joined_1.join(
    df_category,
    on="category_id",
    how="inner"
)

df_children = df_joined_2.filter(col("name") == "Children")
print("Отфильтрованы фильмы только категории 'Children'.")

df_actor_counts = df_children.groupBy("actor_id") \
                             .agg(count(col("film_id")).alias("film_count"))

df_with_names = df_actor_counts.join(
    df_actor,
    on="actor_id",
    how="inner"
)

window_spec = Window.orderBy(col("film_count").desc())

df_ranked = df_with_names.withColumn("rank", rank().over(window_spec))

df_top_3 = df_ranked.filter(col("rank") <= 3) \
                    .withColumn("actor_name", concat_ws(" ", col("first_name"), col("last_name"))) \
                    .select("rank", "actor_name", "film_count") \
                    .orderBy("rank", col("film_count").desc()) 
    
print("Анализ, ранжирование и выбор топ-3 завершены.")

print("\n РЕЗУЛЬТАТ: Топ-3 актера в категории 'Children':")

df_top_3.show(df_top_3.count(), truncate=False)

Отфильтрованы фильмы только категории 'Children'.
Анализ, ранжирование и выбор топ-3 завершены.

 РЕЗУЛЬТАТ: Топ-3 актера в категории 'Children':
+----+-------------+----------+
|rank|actor_name   |film_count|
+----+-------------+----------+
|1   |HELEN VOIGHT |7         |
|2   |WHOOPI HURT  |5         |
|2   |KEVIN GARLAND|5         |
|2   |RALPH CRUZ   |5         |
|2   |MARY TANDY   |5         |
+----+-------------+----------+



In [13]:
# Вывести города с количеством активных и неактивных клиентов (active - customer.active = 1). 
# Сортировать по количеству неактивных клиентов в порядке убывания. 

df_joined_1 = df_customer.join(
    df_address,
    on="address_id", 
    how="inner"
)

df_final_join = df_joined_1.join(
    df_city,
    on="city_id",
    how="inner"
).select("city", "active") 
print("Объединение трех таблиц завершено.")

df_counts = df_final_join.groupBy("city") \
                         .agg(
                             sum(when(col("active") == 1, 1).otherwise(0)).alias("active_customers"),
                             sum(when(col("active") == 0, 1).otherwise(0)).alias("inactive_customers")
                         )

df_sorted = df_counts.orderBy(col("inactive_customers").desc())

print("Агрегация и сортировка завершены.")


print("\nРЕЗУЛЬТАТ: Города с количеством активных и неактивных клиентов (сортировка по неактивным):")

df_sorted.show(10, truncate=False)

Объединение трех таблиц завершено.
Агрегация и сортировка завершены.

РЕЗУЛЬТАТ: Города с количеством активных и неактивных клиентов (сортировка по неактивным):
+----------------+----------------+------------------+
|city            |active_customers|inactive_customers|
+----------------+----------------+------------------+
|Uluberia        |0               |1                 |
|Wroclaw         |0               |1                 |
|Najafabad       |0               |1                 |
|Pingxiang       |0               |1                 |
|Xiangfan        |0               |1                 |
|Kumbakonam      |0               |1                 |
|Szkesfehrvr     |0               |1                 |
|Charlotte Amalie|0               |1                 |
|Kamyin          |0               |1                 |
|Daxian          |0               |1                 |
+----------------+----------------+------------------+
only showing top 10 rows



In [ ]:
# Выведите категорию фильмов с наибольшим общим количеством часов проката в городах (customer.address_id в этом городе),
# начинающихся на букву «a». Сделайте то же самое для городов с символом «-».

df_movie_details = df_rental.join(df_inventory, on="inventory_id") \
                            .join(df_film, on="film_id") \
                            .join(df_film_category, on="film_id") \
                            .join(df_category, on="category_id") \
                            .select("rental_id", "customer_id", "name", "rental_duration")

df_location_details = df_rental.join(df_customer, on="customer_id") \
                               .join(df_address, on="address_id") \
                               .join(df_city, on="city_id") \
                               .select("rental_id", "city")

df_final = df_movie_details.join(df_location_details, on="rental_id")

print("Финальное объединение данных завершено.")


df_with_flag_A = df_final.withColumn(
    "is_A_city",
    when(lower(substring(col("city"), 1, 1)) == 'a', 1).otherwise(0)
)

df_result_A = df_with_flag_A.groupBy("name") \
                            .agg(
                                sum(when(col("is_A_city") == 1, col("rental_duration")).otherwise(0)).alias("total_duration_A")
                            ) \
                            .orderBy(col("total_duration_A").desc()) \
                            .limit(1)

print("\n\n*** РЕЗУЛЬТАТ: ГОРОДА НА 'A' ***")
df_result_A.show(truncate=False)


df_with_flag_dash = df_final.withColumn(
    "is_dash_city",
    when(col("city").contains("-"), 1).otherwise(0)
)

df_result_dash = df_with_flag_dash.groupBy("name") \
                                  .agg(
                                      sum(when(col("is_dash_city") == 1, col("rental_duration")).otherwise(0)).alias("total_duration_DASH")
                                  ) \
                                  .orderBy(col("total_duration_DASH").desc()) \
                                  .limit(1)

print("\n\n*** РЕЗУЛЬТАТ: ГОРОДА С '-' ***")
df_result_dash.show(truncate=False)

spark.stop()
print("\nПрограмма завершена.")